# SageMaker Autopilot — Fintech Credit Risk Demo

**Repo:** `sagemaker-autopilot-demo`  
**Author:** SecuredPress LLC  
**Target:** Predict cash advance repayment (binary classification)  
**Baseline repayment rate:** 88%  

---

### What this notebook does

1. Loads the synthetic fintech dataset and runs exploratory analysis
2. Prepares and uploads training data to S3
3. Triggers a SageMaker Autopilot experiment
4. Evaluates the leaderboard and selects the best model
5. Runs SHAP explainability on the winning candidate
6. Deploys the model to a real-time SageMaker endpoint
7. Runs a sample inference against the live endpoint

> **Prerequisites:** Run `terraform apply` and `bash scripts/upload_training_data.sh` before starting this notebook.

## 1. Setup

Read Terraform outputs and configure the session.

In [ ]:
import subprocess
import json
import boto3
import sagemaker
from sagemaker import AutoML
from sagemaker.automl.automl import AutoMLInput

def tf_output(key):
    """Read a value from terraform output."""
    result = subprocess.run(
        ["terraform", "output", "-raw", key],
        capture_output=True, text=True, check=True,
        cwd="../terraform"
    )
    return result.stdout.strip()

ROLE_ARN            = tf_output("sagemaker_execution_role_arn")
TRAINING_BUCKET     = tf_output("training_bucket_name")
ARTIFACTS_BUCKET    = tf_output("artifacts_bucket_name")
MODEL_PACKAGE_GROUP = tf_output("model_package_group_name")

REGION          = "us-east-1"
JOB_NAME        = tf_output("model_package_group_name").replace("-models", "-job")
TARGET_COL      = "repaid"
TRAIN_S3_URI    = f"s3://{TRAINING_BUCKET}/data/train.csv"
OUTPUT_S3_URI   = f"s3://{ARTIFACTS_BUCKET}/autopilot/"

session = sagemaker.Session()
sm      = boto3.client("sagemaker", region_name=REGION)

print(f"Role ARN:            {ROLE_ARN}")
print(f"Training bucket:     {TRAINING_BUCKET}")
print(f"Artifacts bucket:    {ARTIFACTS_BUCKET}")
print(f"Autopilot job name:  {JOB_NAME}")
print(f"Model package group: {MODEL_PACKAGE_GROUP}")

## 2. Exploratory Data Analysis

Understand the dataset before handing it to Autopilot.  
Key questions: class balance, missing values, feature distributions, correlation with target.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/train.csv")

print(f"Shape: {df.shape}")
print(f"\nTarget distribution:")
print(df["repaid"].value_counts(normalize=True).mul(100).round(1).astype(str) + "%")
print(f"\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# class balance
counts = df["repaid"].value_counts()
axes[0].bar(["Repaid (1)", "Default (0)"], counts.values, color=["#00E676", "#FF4560"])
axes[0].set_title("Class Distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, f"{v:,}", ha="center", fontsize=11)

# balance/advance ratio by class — top predictor
df[df["repaid"] == 1]["balance_advance_ratio"].hist(
    ax=axes[1], bins=40, alpha=0.6, color="#00E676", label="Repaid"
)
df[df["repaid"] == 0]["balance_advance_ratio"].hist(
    ax=axes[1], bins=40, alpha=0.6, color="#FF4560", label="Default"
)
axes[1].set_title("Balance / Advance Ratio by Class")
axes[1].set_xlabel("Ratio")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# numeric feature correlation with target
numeric_cols = df.select_dtypes(include="number").columns.tolist()
corr = df[numeric_cols].corr()["repaid"].drop("repaid").sort_values(ascending=False)

plt.figure(figsize=(9, 5))
colors = ["#00E676" if v > 0 else "#FF4560" for v in corr.values]
plt.barh(corr.index, corr.values, color=colors)
plt.axvline(0, color="white", linewidth=0.8, linestyle="--")
plt.title("Feature Correlation with Target (repaid)")
plt.xlabel("Pearson Correlation")
plt.tight_layout()
plt.show()

print("\nTop 5 positive correlations:")
print(corr.head(5).round(3))
print("\nTop 5 negative correlations:")
print(corr.tail(5).round(3))

## 3. Class Imbalance

The dataset has an 88/12 split. Autopilot handles class imbalance internally during preprocessing,  
but we document the scale of the problem here so the leaderboard metrics are interpreted correctly.

**What this means for evaluation:**  
A naive model that predicts *always repaid* achieves 88% accuracy. AUC-ROC and PR-AUC are the  
correct metrics — they account for class imbalance. Accuracy alone is misleading here.

In [ ]:
from collections import Counter

counter = Counter(df["repaid"])
total   = len(df)

print(f"Class 1 (repaid):   {counter[1]:,}  ({counter[1]/total*100:.1f}%)")
print(f"Class 0 (default):  {counter[0]:,}  ({counter[0]/total*100:.1f}%)")
print(f"Imbalance ratio:    {counter[1]/counter[0]:.1f}:1")
print()
print("Autopilot applies SMOTE oversampling to the minority class during preprocessing.")
print("The raw CSV is passed as-is — do not pre-balance before uploading to S3.")

## 4. SageMaker Autopilot Experiment

Configure and trigger the Autopilot job.  

**Runtime:** 2–4 hours depending on `max_runtime_per_training_job_in_seconds`.  
The cell below starts the job and returns immediately — monitor progress in the next cell  
or in the AWS console under **SageMaker → Autopilot → Jobs**.

> This cell is idempotent — re-running it will raise a `ResourceInUse` error if the job  
> already exists, which is safe to ignore.

In [ ]:
automl = AutoML(
    role=ROLE_ARN,
    target_attribute_name=TARGET_COL,
    output_path=OUTPUT_S3_URI,
    problem_type="BinaryClassification",
    max_auto_ml_runtime_in_seconds=14400,       # 4 hours — matches terraform variable
    max_runtime_per_training_job_in_seconds=600,# 10 min per individual trial
    job_objective={"MetricName": "AUC"},
    generate_candidate_definitions_only=False,
    sagemaker_session=session,
    base_job_name=JOB_NAME,
    model_deploy_config={
        "AutoGenerateEndpointName": False,
        "EndpointName": "",
    }
)

train_input = AutoMLInput(
    inputs=TRAIN_S3_URI,
    target_attribute_name=TARGET_COL,
    compression=None,
)

try:
    automl.fit(
        inputs=train_input,
        job_name=JOB_NAME,
        wait=False,   # non-blocking — monitor with the cell below
        logs=False,
    )
    print(f"Autopilot job started: {JOB_NAME}")
    print("Monitor in AWS console: SageMaker → Autopilot → Jobs")
except Exception as e:
    if "ResourceInUse" in str(e) or "already exists" in str(e).lower():
        print(f"Job already exists: {JOB_NAME} — skipping.")
    else:
        raise

In [ ]:
# poll job status — run this cell periodically while waiting
# or monitor the AWS console

response = sm.describe_auto_ml_job(AutoMLJobName=JOB_NAME)
status   = response["AutoMLJobStatus"]
sec_run  = response.get("AutoMLJobArtifacts", {}).get("CandidateDefinitionNotebookLocation", "—")

print(f"Job name:  {JOB_NAME}")
print(f"Status:    {status}")

if status == "Completed":
    best = response.get("BestCandidate", {})
    metric = best.get("FinalAutoMLJobObjectiveMetric", {})
    print(f"Best AUC:  {metric.get('Value', 'n/a')}")
    print(f"Algorithm: {best.get('CandidateName', 'n/a')}")
elif status in ["InProgress", "Stopped", "Failed"]:
    secondary = response.get("AutoMLJobSecondaryStatus", "")
    print(f"Secondary: {secondary}")
    if status == "Failed":
        print(f"Reason:    {response.get('FailureReason', 'n/a')}")

## 5. Leaderboard

Review all candidates ranked by AUC-ROC.  
Run this cell after the job status shows `Completed`.

In [ ]:
import pandas as pd

candidates = []
paginator  = sm.get_paginator("list_candidates_for_auto_ml_job")

for page in paginator.paginate(
    AutoMLJobName=JOB_NAME,
    SortBy="FinalObjectiveMetricValue",
    SortOrder="Descending"
):
    candidates.extend(page["Candidates"])

rows = []
for c in candidates:
    metric = c.get("FinalAutoMLJobObjectiveMetric", {})
    rows.append({
        "Candidate":  c["CandidateName"],
        "Status":     c["CandidateStatus"],
        "AUC-ROC":    round(metric.get("Value", 0), 4),
    })

leaderboard = pd.DataFrame(rows)
leaderboard.index = leaderboard.index + 1  # rank from 1

baseline_auc = 0.874
best_auc     = leaderboard["AUC-ROC"].max()
improvement  = round((best_auc - baseline_auc) / baseline_auc * 100, 1)

print(f"Total candidates:  {len(leaderboard)}")
print(f"Best AUC-ROC:      {best_auc}")
print(f"Baseline AUC-ROC:  {baseline_auc}")
print(f"Improvement:       +{improvement}%")
print()
leaderboard.head(10)

## 6. Register Best Model

Register the winning candidate in the SageMaker Model Registry for governance and deployment.  
Approval status is set to `Approved` so Terraform can deploy it in Phase 3.

In [ ]:
best_candidate_name = leaderboard.iloc[0]["Candidate"]
print(f"Registering: {best_candidate_name}")

# retrieve the best candidate object
best_candidate = next(
    c for c in candidates if c["CandidateName"] == best_candidate_name
)

# register to the model package group provisioned by Terraform
response = sm.create_model_package(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    ModelPackageDescription=(
        f"Best Autopilot candidate — AUC-ROC {leaderboard.iloc[0]['AUC-ROC']}"
    ),
    InferenceSpecification={
        "Containers": best_candidate["InferenceContainers"],
        "SupportedContentTypes":     ["text/csv"],
        "SupportedResponseMIMETypes": ["text/csv"],
    },
    ModelApprovalStatus="Approved",
)

model_package_arn = response["ModelPackageArn"]
print(f"Registered:  {model_package_arn}")
print()
print("Add this ARN to terraform/terraform.tfvars:")
print(f'  model_package_arn = "{model_package_arn}"')
print()
print("Then uncomment endpoint.tf and run: terraform apply")

## 7. Feature Importance — SHAP

Run SHAP explainability on a local version of the best model.  
This produces the feature importance chart used in the dashboard.

> SHAP runs locally against XGBoost — no endpoint needed for this step.

In [ ]:
import shap
import xgboost as xgb
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("../data/train.csv")

# encode categoricals
df_enc = df.copy()
for col in ["device_os"]:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col].astype(str))

X = df_enc.drop(columns=["repaid"])
y = df_enc["repaid"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric="auc",
    random_state=42,
    verbosity=0,
)
model.fit(X_train, y_train)

explainer  = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

print("SHAP values computed.")

In [ ]:
# global feature importance — mean absolute SHAP value
mean_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=X.columns
).sort_values(ascending=True)

plt.figure(figsize=(9, 6))
colors = ["#00E5FF" if v > mean_shap.median() else "#845EF7" for v in mean_shap.values]
plt.barh(mean_shap.index, mean_shap.values, color=colors)
plt.title("Global SHAP Feature Importance")
plt.xlabel("Mean |SHAP value|")
plt.tight_layout()
plt.show()

print("\nTop 5 features:")
print(mean_shap.tail(5).round(4))

In [ ]:
# beeswarm — shows direction of impact, not just magnitude
shap.summary_plot(shap_values, X_test, max_display=10, show=True)

## 8. Sample Inference

Once the endpoint is deployed (Phase 3 — after uncommenting `endpoint.tf` and running  
`terraform apply`), test it with a real-time prediction.

Replace `ENDPOINT_NAME` with the value from `terraform output endpoint_name`  
or use `tf_output("endpoint_name")` if the output is wired in `outputs.tf`.

In [ ]:
import boto3
import json

ENDPOINT_NAME = tf_output("endpoint_name")  # from terraform outputs after Phase 3

runtime = boto3.client("sagemaker-runtime", region_name=REGION)

# sample payload — high-confidence repayer
sample = {
    "balance_advance_ratio":  2.4,
    "prior_repay_score":      0.87,
    "days_since_payroll":     3,
    "avg_monthly_income":     3200,
    "tx_velocity_30d":        18,
    "overdraft_frequency":    1,
    "advance_amount":         75,
    "account_balance":        180.0,
    "payroll_interval_days":  14,
    "days_until_payroll":     11,
    "neobank":                0,
    "device_os":              1,   # encoded: iOS
    "institution_id":         1001,
    "age":                    29,
    "months_as_customer":     18,
    "prior_advance_count":    4,
    "last_repay_delay_days":  2,
    "income_stability":       0.81,
}

# SageMaker Autopilot endpoints expect CSV input
payload = ",".join(str(v) for v in sample.values())

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="text/csv",
    Body=payload,
)

result       = response["Body"].read().decode("utf-8").strip()
repay_prob   = float(result)
decision     = "APPROVE" if repay_prob >= 0.50 else "DENY"
advance_bin  = "$75" if repay_prob >= 0.75 else "$50" if repay_prob >= 0.50 else "$0"

print(f"Repayment probability: {repay_prob:.4f}")
print(f"Decision:              {decision}")
print(f"Advance bin:           {advance_bin}")

## 9. Cleanup

Delete just the endpoint to stop hourly charges while keeping the model and Autopilot results.

```bash
# stop endpoint billing only
terraform destroy -target aws_sagemaker_endpoint.demo

# full teardown — removes all resources including S3 buckets and model artifacts
terraform destroy
```

> Running `terraform destroy` without `-target` removes everything including training data  
> and Autopilot artifacts from S3. Only use it when you are done with the demo entirely.